# Lab 3b — THE LINE
**هندسة تطبيقات النماذج اللغوية الكبيرة**  
**إعداد: أ. لطيفة سعد**

## ⓪ PROJECT SETTINGS — إعدادات مشروعك

In [1]:
ASSISTANT_NAME = "THE LINE"

CURRENT_USER = "Awadh"

APPLICATIONS = {
    "ENG12345678": "under review",
    "ENG87654321": "approved"
}

print("Assistant   :", ASSISTANT_NAME)
print("Current user:", CURRENT_USER)
print("Applications:", APPLICATIONS)

Assistant   : THE LINE
Current user: Awadh
Applications: {'ENG12345678': 'under review', 'ENG87654321': 'approved'}


## ① TOOL REGISTRY — الأدوات الثلاث وسجلها

In [2]:
def check_application_status(reference):
    return APPLICATIONS.get(reference, "not found")

def book_appointment(name, city, date):
    return "Engineering consultation confirmed for " + name + " in " + city + " on " + date

def escalate_to_agent(reason):
    return "Transferred to a project support engineer: " + reason

TOOLS = {
    "check_application_status": {
        "function": check_application_status,
        "risk": "read_only"
    },
    "book_appointment": {
        "function": book_appointment,
        "risk": "side_effecting"
    },
    "escalate_to_agent": {
        "function": escalate_to_agent,
        "risk": "terminal"
    }
}

for name, tool in TOOLS.items():
    print(name, "→", tool["risk"])

check_application_status → read_only
book_appointment → side_effecting
escalate_to_agent → terminal


## ② AUTHORIZATION GATE — بوابة الصلاحية

In [3]:
def run_tool(name, args):
    if name not in TOOLS:
        return "error: unknown tool"

    tool = TOOLS[name]

    if tool["risk"] == "side_effecting" and args.get("name") != CURRENT_USER:
        return "error: not authorized"

    try:
        return tool["function"](**args)
    except TypeError:
        return "error: wrong arguments"

In [4]:
print(run_tool("check_application_status", {"reference": "ENG12345678"}))

booking = {"name": "Awadh", "city": "Tabuk", "date": "2026-10-10"}
print(run_tool("book_appointment", booking))

booking = {"name": "Fahad", "city": "Tabuk", "date": "2026-10-10"}
print(run_tool("book_appointment", booking))

under review
Engineering consultation confirmed for Awadh in Tabuk on 2026-10-10
error: not authorized


## ③ BOUNDED TOOL LOOP — حلقة الأدوات المحدودة

In [5]:
MAX_ITERATIONS = 6

def run_with_tools(steps):
    for iteration in range(1, MAX_ITERATIONS + 1):
        step = steps[iteration - 1]

        if "text" in step:
            return step["text"]

        result = run_tool(step["tool"], step["args"])
        print("Iteration", iteration, "|", step["tool"], "→", result)

    return "I could not complete this. Transferring you to a project support engineer."

## ④ TEST: BOOKING AND NEGATIVE TESTS — اختبار الحجز والاختبارات السلبية

In [6]:
steps = [
    {
        "tool": "check_application_status",
        "args": {"reference": "ENG12345678"}
    },
    {
        "tool": "book_appointment",
        "args": {"name": "Awadh", "city": "Tabuk", "date": "2026-10-10"}
    },
    {
        "text": "Your sustainability engineering request is under review, and your consultation appointment is booked."
    }
]

print(run_with_tools(steps))

Iteration 1 | check_application_status → under review
Iteration 2 | book_appointment → Engineering consultation confirmed for Awadh in Tabuk on 2026-10-10
Your sustainability engineering request is under review, and your consultation appointment is booked.


In [7]:
status_call = {
    "tool": "check_application_status",
    "args": {"reference": "ENG12345678"}
}

fahad_booking = {"name": "Fahad", "city": "Tabuk", "date": "2026-10-10"}

bad_steps = [
    {"tool": "delete_everything", "args": {}},
    {"tool": "book_appointment", "args": fahad_booking},
    {"tool": "check_application_status", "args": {"number": "ENG12345678"}},
    status_call,
    status_call,
    status_call,
    status_call
]

print(run_with_tools(bad_steps))

Iteration 1 | delete_everything → error: unknown tool
Iteration 2 | book_appointment → error: not authorized
Iteration 3 | check_application_status → error: wrong arguments
Iteration 4 | check_application_status → under review
Iteration 5 | check_application_status → under review
Iteration 6 | check_application_status → under review
I could not complete this. Transferring you to a project support engineer.
